# 实践 16.1  0.5×、1×与2×模拟测序的后验剂量估计

《统计遗传与智能育种学》第十六章 · 电子资源 `16_lowcov_imputation.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：16_lowcov_imputation.ipynb。本实践把参考群与目标群严格分开：参考群只用于估计等位基因频率并构造HWE基因型先验，目标群真实基因型不参与先验估计；再用Poisson深度、1%对称测序错误和二项读段抽样构造0.5×、1×和2×数据，对g=0/1/2计算基因型似然与后验剂量。该实现属于单个位点贝叶斯基线，未利用LD或参考单倍型，不能替代正式基因型填充。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果见下方代码输出。指标不预设严格单调；解释时必须区分平均深度、至少1条读段覆盖的比例、逐SNP后验剂量r²和硬判定基因型一致率。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1601；320个参考群个体、240个目标群个体、300个模拟SNP；测序错误率1%；170/70训练—测试划分；无外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
from scipy.special import logsumexp
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 1601
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

# 参考群只用于估计等位基因频率；目标群真值绝不参与先验估计。
n_ref, n_target, m = 320, 240, 300
pop_p = rng.uniform(0.05, 0.50, m)
G_ref = rng.binomial(2, pop_p, size=(n_ref, m))
G_true = rng.binomial(2, pop_p, size=(n_target, m))
p_ref = np.clip(G_ref.mean(axis=0) / 2, 0.01, 0.99)
SEQ_ERROR = 0.01

def sitewise_posterior(depth, rng_local):
    # Poisson读深；给定二倍体基因型g，替代等位基因读段概率含1%对称测序错误。
    d = rng_local.poisson(depth, size=G_true.shape)
    q_true = SEQ_ERROR + (1 - 2 * SEQ_ERROR) * G_true / 2
    alt = rng_local.binomial(d, q_true)

    # HWE先验只来自独立参考群。g=0/1/2的读段概率分别为e/0.5/(1-e)。
    prior = np.stack([(1-p_ref)**2, 2*p_ref*(1-p_ref), p_ref**2], axis=1)
    q = np.array([SEQ_ERROR, 0.5, 1-SEQ_ERROR])
    logpost = np.empty((n_target, m, 3))
    for g in range(3):
        loglik = alt*np.log(q[g]) + (d-alt)*np.log(1-q[g])
        logpost[:, :, g] = loglik + np.log(prior[:, g])[None, :]
    logpost -= logsumexp(logpost, axis=2, keepdims=True)
    post = np.exp(logpost)
    dosage = post[:, :, 1] + 2*post[:, :, 2]
    hard = np.argmax(post, axis=2)
    max_p = post.max(axis=2)
    return d, dosage, hard, max_p

def per_snp_dosage_r2(dosage, truth):
    # 对每个SNP在个体间计算剂量r²；单态或无剂量变异的SNP记为缺失。
    out = np.full(truth.shape[1], np.nan)
    for j in range(truth.shape[1]):
        if np.std(truth[:, j]) > 0 and np.std(dosage[:, j]) > 0:
            rj = np.corrcoef(dosage[:, j], truth[:, j])[0, 1]
            if np.isfinite(rj):
                out[j] = rj * rj
    return out

rows = []
cache = {}
for dep in [0.5, 1.0, 2.0]:
    # 每个深度使用独立、但可复现的随机流。
    d, dosage, hard, max_p = sitewise_posterior(dep, np.random.default_rng(SEED + int(dep*100)))
    cache[dep] = dosage
    r2_snp = per_snp_dosage_r2(dosage, G_true)
    rows.append({
        'depth_x': dep,
        'mean_depth': d.mean(),
        'covered_ge1': (d >= 1).mean(),
        'zero_depth': (d == 0).mean(),
        'mean_snp_dosage_r2': np.nanmean(r2_snp),
        'median_snp_dosage_r2': np.nanmedian(r2_snp),
        'hard_concordance': (hard == G_true).mean(),
        'uncertain_Pmax_lt0.9': (max_p < 0.9).mean(),
    })
metrics = pd.DataFrame(rows)
print(metrics.round(4).to_string(index=False))

# 同一数量性状、同一训练/测试划分比较真实基因型与低深度后验剂量。
beta = rng.normal(0, 0.10, m)
y = G_true @ beta + rng.normal(0, 1.0, n_target)
tr = np.arange(170)
te = np.arange(170, n_target)

def ridge_r(X):
    model = make_pipeline(StandardScaler(), Ridge(alpha=80.0))
    model.fit(X[tr], y[tr])
    return np.corrcoef(model.predict(X[te]), y[te])[0, 1]

print('true_genotype_Ridge_test_r', round(ridge_r(G_true), 3))
for dep in [0.5, 1.0, 2.0]:
    print('depth', dep, 'sitewise_dosage_Ridge_test_r', round(ridge_r(cache[dep]), 3))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1601
 depth_x  mean_depth  covered_ge1  zero_depth  mean_snp_dosage_r2  median_snp_dosage_r2  hard_concordance  uncertain_Pmax_lt0.9
     0.5      0.4966       0.3922      0.6078              0.2008                0.2002            0.6678                0.9157
     1.0      0.9934       0.6277      0.3723              0.3498                0.3492            0.7194                0.8199
     2.0      2.0002       0.8634      0.1366              0.5646                0.5692            0.8033                0.6178
true_genotype_Ridge_test_r 0.371
depth 0.5 sitewise_dosage_Ridge_test_r 0.182
depth 1.0 sitewise_dosage_Ridge_test_r 0.289
depth 2.0 sitewise_dosage_Ridge_test_r 0.329


结果解释：0.5×、1×和2×条件下，至少1条读段覆盖的比例与Poisson抽样机制相符，但不等于平均深度。后验剂量r²先在每个SNP上跨个体计算，再跨SNP汇总，各位点等权汇总，减少合并位点时频率差异的影响；硬判定基因型一致率评价的是另一对象。零覆盖位点在本代码中只能依据参考群频率先验估计；由于没有参考单倍型隐马尔可夫模型（hidden Markov model, HMM），该实现仅为单个位点基线。Ridge结果用于说明测量误差会传播到下游预测，不能据此推荐任何实际物种的测序深度。

可修改练习：把SEQ_ERROR改为0.005和0.02；把参考群规模由320降为50；分别观察0.5×时mean_snp_dosage_r2和Ridge test r如何变化，并解释先验估计误差与读段证据的相对作用。